# Static CNN/ResNet under kappa shift

这里先用固定状态 `kappa=0` 的样本训练 static CNN/ResNet，再用同一个模型测试不同 `kappa`。shifted data 只用于测试，不参与训练。

`kappa` 越大，相当于测试时的光纤状态和训练时的固定状态偏得越远。这个实验用来说明 static inverse mapping 的局限。

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "MMF_dataset"
if not DATA_DIR.exists():
    raise FileNotFoundError("Cannot find MMF dataset folder.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", DEVICE)
print("data folder:", DATA_DIR.name)

In [ ]:
def load_dataset(data_dir):
    return {
        "pattern": np.load(data_dir / "pattern.npy", mmap_mode="r"),
        "speckles": np.load(data_dir / "speckles.npy", mmap_mode="r"),
        "time_index": np.load(data_dir / "time_index.npy", mmap_mode="r"),
        "kappa_by_time": np.load(data_dir / "kappa_by_time.npy"),
        "metadata": json.loads((data_dir / "metadata.json").read_text(encoding="utf-8")),
    }


def split_static_ids(time_index, seed=42):
    # 这里只用 kappa=0 的样本训练，测试集不会参与模型更新
    rng = np.random.default_rng(seed)
    ids = np.where(time_index == 0)[0].copy()
    rng.shuffle(ids)
    return ids[:80000], ids[80000:90000], ids[90000:100000]


arrays = load_dataset(DATA_DIR)
train_ids, val_ids, test_ids = split_static_ids(arrays["time_index"])
assert len(set(train_ids) & set(val_ids)) == 0
assert len(set(train_ids) & set(test_ids)) == 0
assert len(set(val_ids) & set(test_ids)) == 0

In [ ]:
class MMFDataset(Dataset):
    def __init__(self, speckles, patterns, indices):
        self.speckles = speckles
        self.patterns = patterns
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = int(self.indices[item])
        x = self.speckles[idx].astype(np.float32) / 255.0
        x = (x - x.mean()) / (x.std() + 1e-6)
        y = self.patterns[idx].astype(np.float32)
        return torch.from_numpy(x).unsqueeze(0), torch.from_numpy(y)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)


class StaticResNetCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            DownBlock(1, 32),
            ResidualBlock(32, dropout=0.02),
            DownBlock(32, 64),
            ResidualBlock(64, dropout=0.03),
            ResidualBlock(64, dropout=0.03),
            DownBlock(64, 128),
            ResidualBlock(128, dropout=0.05),
            ResidualBlock(128, dropout=0.05),
            DownBlock(128, 192),
            ResidualBlock(192, dropout=0.05),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192 * 4 * 4, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 256),
        )

    def forward(self, x):
        return self.head(self.features(x))

In [ ]:
def sample_corr(pred, target, eps=1e-8):
    pred = pred - pred.mean(dim=1, keepdim=True)
    target = target - target.mean(dim=1, keepdim=True)
    numerator = (pred * target).sum(dim=1)
    denominator = torch.sqrt(pred.square().sum(dim=1) * target.square().sum(dim=1) + eps)
    return numerator / denominator


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_n = 0
    totals = {"loss": 0.0, "pixel_acc": 0.0, "mse": 0.0, "corr": 0.0}
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(x)
        prob = torch.sigmoid(logits)
        batch_size = x.size(0)
        totals["loss"] += F.binary_cross_entropy_with_logits(logits, y).item() * batch_size
        totals["pixel_acc"] += ((prob > 0.5) == y.bool()).float().mean().item() * batch_size
        totals["mse"] += (prob - y).square().mean().item() * batch_size
        totals["corr"] += sample_corr(prob, y).mean().item() * batch_size
        total_n += batch_size
    return {name: value / total_n for name, value in totals.items()} | {"n": total_n}

In [ ]:
def train_static_cnn():
    # 这里训练的是 static CNN，只学习一个固定光纤状态下的逆映射
    torch.manual_seed(42)
    train_loader = DataLoader(MMFDataset(arrays["speckles"], arrays["pattern"], train_ids), batch_size=128, shuffle=True)
    val_loader = DataLoader(MMFDataset(arrays["speckles"], arrays["pattern"], val_ids), batch_size=128)
    model = StaticResNetCNN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)
    best_corr = -1.0
    best_state = None
    history = []

    for epoch in range(1, 16):
        model.train()
        total_loss = 0.0
        total_n = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = F.binary_cross_entropy_with_logits(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            total_n += x.size(0)
        scheduler.step()
        val = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": total_loss / total_n, **val})
        if val["corr"] > best_corr:
            best_corr = val["corr"]
            best_state = {name: value.cpu().clone() for name, value in model.state_dict().items()}
        print(f"epoch {epoch:02d} | val_acc={val['pixel_acc']:.4f} | val_corr={val['corr']:.4f}")

    model.load_state_dict(best_state)
    return model, history

In [ ]:
# 先训练一个固定状态下的 static CNN，再测试 kappa shift
model, history = train_static_cnn()

In [ ]:
# 用训练好的模型直接测不同 kappa 下的表现，不在 shifted data 上重新训练
rows = []
for state, kappa in enumerate(arrays["kappa_by_time"]):
    ids = np.where(arrays["time_index"] == state)[0]
    if len(ids) > 5000:
        rng = np.random.default_rng(2000 + state)
        ids = rng.choice(ids, size=5000, replace=False)
    loader = DataLoader(MMFDataset(arrays["speckles"], arrays["pattern"], ids), batch_size=256)
    metrics = evaluate(model, loader)
    metrics["wrong_pixels_per_pattern"] = (1.0 - metrics["pixel_acc"]) * 256
    rows.append({"time_index": state, "kappa": float(kappa), **metrics})

static = rows[0]
for row in rows:
    row["delta_acc_vs_static"] = row["pixel_acc"] - static["pixel_acc"]
    row["delta_corr_vs_static"] = row["corr"] - static["corr"]
    print(
        f"kappa={row['kappa']:.1f} | acc={row['pixel_acc']:.4f} | "
        f"corr={row['corr']:.4f} | wrong={row['wrong_pixels_per_pattern']:.1f}/256"
    )

In [ ]:
# 主图直接显示 static CNN/ResNet 在不同 kappa 下的变化
kappas = [row["kappa"] for row in rows]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(kappas, [row["pixel_acc"] for row in rows], "o-")
axes[0].set_title("accuracy vs kappa")
axes[0].set_ylabel("pixel accuracy")
axes[1].plot(kappas, [row["corr"] for row in rows], "o-")
axes[1].set_title("correlation vs kappa")
axes[1].set_ylabel("probability-label corr")
axes[2].plot(kappas, [row["wrong_pixels_per_pattern"] for row in rows], "o-")
axes[2].set_title("wrong pixels per pattern")
axes[2].set_ylabel("wrong / 256")
for axis in axes:
    axis.set_xlabel("kappa")
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# 拼一张图直观看 static model 随 kappa 偏移后的重建变化
states = [0, 3, 5, 10]
fig, axes = plt.subplots(len(states), 4, figsize=(9, 2.3 * len(states)))
with torch.no_grad():
    for row, state in enumerate(states):
        ids = np.where(arrays["time_index"] == state)[0]
        rng = np.random.default_rng(223 + state)
        idx = int(rng.choice(ids))
        x, y = MMFDataset(arrays["speckles"], arrays["pattern"], [idx])[0]
        prob = torch.sigmoid(model(x.unsqueeze(0).to(DEVICE)))[0].cpu().numpy().reshape(16, 16)
        pred = (prob > 0.5).astype(np.float32)
        label = y.numpy().reshape(16, 16)
        error = (pred != label).astype(np.float32)
        acc = float((pred == label).mean())
        corr = float(np.corrcoef(prob.ravel(), label.ravel())[0, 1])
        kappa = float(arrays["kappa_by_time"][state])

        images = [x[0].numpy(), pred, label, error]
        titles = [f"speckle\nkappa={kappa:.1f}", f"prediction\nacc={acc:.3f}", "label", f"error\ncorr={corr:.3f}"]
        cmaps = ["inferno", "gray", "gray", "Reds"]
        for col in range(4):
            axes[row, col].imshow(images[col], cmap=cmaps[col], vmin=0 if col else None, vmax=1 if col else None)
            axes[row, col].set_title(titles[col])
            axes[row, col].axis("off")

plt.tight_layout()
plt.show()